In [ ]:
## Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import BertTokenizer,BertForSequenceClassification
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
## Loading validation data
val_data=pd.read_csv('/content/validation.csv')

In [ ]:
## Dimensions of validation data
val_data.shape

In [ ]:
## Basic information of data
val_data.info()

In [ ]:
## First 5 rows of data
val_data.head()

In [ ]:
## Form validation pairs
sent_pairs=[(row['sentence1'],row['sentence2']) for index,row in val_data.iterrows()]
len(sent_pairs)

In [ ]:
sent_pairs[1]

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
## Load tokenizer
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
## Tokenization
def tokenization(sent1, sent2):
    encoded = tokenizer.encode_plus(
        sent1,sent2,
        add_special_tokens=True,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoded['input_ids']
    attention_masks = encoded['attention_mask']
    return input_ids, attention_masks

In [ ]:
## Prediction of marks and accuracies/similarities
def predict_score_and_accuracy(sent_pairs,model):
  marks_obtained=[]
  similarities = []
  for sent1, sent2 in sent_pairs:
      input_ids, attention_mask = tokenization(sent1, sent2)
      with torch.no_grad():
          outputs = model(input_ids.to(device), attention_mask.to(device))
          logits = outputs.logits
          mark_obtained = torch.sigmoid(logits)
          accuracy = torch.argmax(logits, dim=1).item()
          if accuracy==0:
            avg_marks=torch.min(mark_obtained).item()
          elif accuracy==1:
            avg_marks=torch.mean(mark_obtained).item()
          elif accuracy==2:
            avg_marks=torch.max(mark_obtained).item()
      similarities.append(accuracy)
      marks_obtained.append(avg_marks)
  return similarities, marks_obtained

In [ ]:
## Load fine-tuned saved models
model1=BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/model11').to(device)
model2=BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/model21').to(device)
model3=BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/model31').to(device)

In [ ]:
## Prediction
sim1,scores1=predict_score_and_accuracy(sent_pairs,model1)

In [ ]:
## Classification report for model1
report1=classification_report(val_data['similarity'],sim1)
rep_lines=report1.split('\n')
rep_lines=rep_lines[2:]
df=[]
for line in rep_lines:
  if line.strip():
    row=line.strip().split()
    df.append({'class':row[0],'precision':row[1],'recall':row[2]})
print('Classifiction Report for Model 1'+'\n',pd.DataFrame(df))

In [ ]:
## Prediction
sim2,scores2=predict_score_and_accuracy(sent_pairs,model2)

In [ ]:
## Classification report for model2
report2=classification_report(val_data['similarity'],sim2)
rep_lines=report2.split('\n')
rep_lines=rep_lines[2:]
df=[]
for line in rep_lines:
  if line.strip():
    row=line.strip().split()
    df.append({'class':row[0],'precision':row[1],'recall':row[2]})
print('Classifiction Report for Model 2'+'\n',pd.DataFrame(df))

In [ ]:
## Prediction
sim3,scores3=predict_score_and_accuracy(sent_pairs,model3)

In [ ]:
## Classification report for model3
report3=classification_report(val_data['similarity'],sim3)
rep_lines=report3.split('\n')
rep_lines=rep_lines[2:]
df=[]
for line in rep_lines:
  if line.strip():
    row=line.strip().split()
    df.append({'class':row[0],'precision':row[1],'recall':row[2]})
print('Classifiction Report for Model 3'+'\n',pd.DataFrame(df))

In [ ]:
## Confusion matrices for all 3 models
cm1=confusion_matrix(val_data['similarity'],sim1)
cm2=confusion_matrix(val_data['similarity'],sim2)
cm3=confusion_matrix(val_data['similarity'],sim3)
fig,axes=plt.subplots(1,3,figsize=(60,20))
sns.heatmap(cm1,annot=True,annot_kws={'size':35},cmap='coolwarm',ax=axes[0])
axes[0].set_title('Confusion Matrix for Model1',fontsize=40)
axes[0].tick_params(axis='both',labelsize=40)
sns.heatmap(cm2,annot=True,annot_kws={'size':35},cmap='coolwarm',ax=axes[1])
axes[1].set_title('Confusion Matrix for Model2',fontsize=40)
axes[1].tick_params(axis='both',labelsize=40)
plt.tick_params(axis='both',labelsize=40)
sns.heatmap(cm3,annot=True,annot_kws={'size':35},cmap='coolwarm',ax=axes[2])
axes[2].set_title('Confusion Matrix for Model3',fontsize=40)
axes[2].tick_params(axis='both',labelsize=40)
plt.tight_layout()
plt.show()